# Lightweight Relational Knowledge Fusion (P2 — Stanceosaurus)

Rút gọn từ **LLM-CRAN** (Niu et al., *MT2-CSD*, arXiv 2506.21053, 2025) — kiến trúc SOTA cho đúng bài toán Conversational Stance Detection đa lượt đa bên, đã chứng minh vượt cả ChatGPT-4/Llama2-70b in-context trên benchmark MT2-CSD (Favg 54.19 vs 48.18/42.51).

**Khác biệt cốt lõi so với SITPCL/FoT (vốn xử lý hội thoại như 1 chuỗi tuần tự qua GRU):**
Thay vì ép toàn bộ hội thoại qua 1 GRU tuần tự (giả định Markov yếu, lượt sau chỉ phụ thuộc trạng thái ẩn tích lũy), pipeline này xây **đồ thị hội thoại thật** (node = lượt nói, cạnh = quan hệ reply + quan hệ discourse) và dùng **GCN/RGCN + multi-hop attention** để tổng hợp 3 loại "tri thức":

1. **Local Knowledge (LKL):** CNN cửa sổ trượt quanh lượt đang xét — nắm ngữ cảnh cục bộ gần nhất.
2. **Contextual/Structural Knowledge (CRL):** GCN trên đồ thị reply-chain thật của hội thoại — nắm quan hệ toàn cục, không bị "quên" như GRU khi chuỗi dài.
3. **Discourse-Relation Knowledge (RGCN):** quan hệ logic giữa các lượt (contrastive/causal/succession/summary), gán nhãn bằng **bộ phân loại rule-based dựa discourse marker** (thay cho gọi GPT-4 trả phí như bản gốc, để chạy offline miễn phí trên Kaggle — đánh đổi: yếu hơn LLM thật, coi là baseline rẻ của ý tưởng, có thể nâng cấp sau bằng LLM thật nếu có API).

Ba luồng tri thức được hợp nhất bằng **multi-hop attention** (lặp lại p=2 vòng), rồi mới phân loại 3 lớp.

> ⚠️ **Đính chính:** `use_evidence_injection` (chèn `title` từ Excel) **mặc định TẮT (False)**. Lý do: 3 pipeline cũ (baseline/SITPCL/SITPCL-FoT) đều chỉ dùng `train_conversations.json` làm input duy nhất — để so sánh Macro-F1 công bằng với bảng tham chiếu, kết quả chính của notebook này phải chạy trên **cùng input đó**, không thêm `title`. Evidence Injection chỉ nên bật như **một thử nghiệm riêng biệt** (so với chính bản tắt, không so thẳng với bảng tham chiếu) — không liên quan đến lý do kiến trúc chính (GCN/RGCN) của P2.

**Lưu ý:** đây là hướng kiến trúc mới, rủi ro/độ phức tạp triển khai cao hơn P1. Nên chạy P1 trước (chi phí thấp, đã có cơ sở chắc), dùng P2 làm thực nghiệm đối chứng táo bạo hơn.

**Yêu cầu Kaggle Input:** chỉ cần **1 dataset bắt buộc** — `Stanceosaurus_reconstructed_dialogues`, giống hệt 3 pipeline cũ. `Stanceosaurus-data-main` (Excel) là **tùy chọn**, chỉ cần khi bật `use_evidence_injection = True`.

In [ ]:
# ==============================================================================
# STEP 1: CÀI ĐẶT THƯ VIỆN & DÒ TÌM DỮ LIỆU
# ==============================================================================
!pip install -q transformers openpyxl emoji python-box scikit-learn

import os
import re
import glob
import json
import math
import random
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm.auto import tqdm
from box import Box

# Nguồn BẮT BUỘC (duy nhất) — giống hệt 3 pipeline cũ, để so sánh công bằng (cùng input).
found_json = glob.glob('/kaggle/input/**/train_conversations.json', recursive=True) + \
             glob.glob('./**/train_conversations.json', recursive=True) + \
             glob.glob('D:/KLTN/**/train_conversations.json', recursive=True)
if not found_json:
    raise FileNotFoundError('❌ Không tìm thấy train_conversations.json!')
JSON_DIR = os.path.dirname(found_json[0])

# Nguồn TÙY CHỌN — chỉ cần nếu bật use_evidence_injection=True ở Cell 2 (thử nghiệm riêng, xem markdown đầu file).
found_xlsx = glob.glob('/kaggle/input/**/Train_final.xlsx', recursive=True) + \
             glob.glob('./**/Train_final.xlsx', recursive=True) + \
             glob.glob('D:/KLTN/**/Train_final.xlsx', recursive=True)
XLSX_DIR = os.path.dirname(found_xlsx[0]) if found_xlsx else None
print(f'--> ✅ JSON: {JSON_DIR} | Excel (tùy chọn): {XLSX_DIR or "không có — evidence injection sẽ tự tắt"}')

In [ ]:
# ==============================================================================
# STEP 2: CẤU HÌNH
# ==============================================================================
DEBUG_MODE = False

config = Box({
    'json_dir': JSON_DIR,
    'xlsx_dir': XLSX_DIR,
    'debug_mode': DEBUG_MODE,
    'debug_samples_per_class': 40,
    'model_name': 'vinai/bertweet-base',
    'num_classes': 3,
    'max_seq_len': 128,
    'batch_size': 8,          # mỗi hội thoại giờ tốn nhiều bộ nhớ hơn (CNN+GCN+RGCN) nên giảm batch
    'accum_steps': 4,
    'learning_rate': 2e-5,
    'gnn_lr': 1e-4,
    'weight_decay': 0.01,
    'num_epochs': 1 if DEBUG_MODE else 25,  # Trần epoch cao, dừng thật bằng Early Stopping (xem early_stop_patience)
    'early_stop_patience': 4,             # Dừng sớm nếu Dev Macro-F1 không cải thiện sau 4 epoch liên tiếp
    'warmup_ratio': 0.1,
    'hidden_dim': 768,
    'cnn_kernel': 3,
    'num_relations_logical': 4,     # contrastive, causal, succession, summary
    'multihop_iters': 2,
    'multihop_lambda': 0.2,

    # Mặc định TẮT — giữ đúng nguyên tắc "cùng input" với 3 pipeline cũ. Chỉ bật True cho một
    # thử nghiệm RIÊNG BIỆT (cần gắn thêm dataset Stanceosaurus-data-main), không dùng để so sánh
    # thẳng với bảng tham chiếu (xem đính chính ở markdown đầu file).
    'use_evidence_injection': False,

    'loss_type': 'focal',
    'focal_gamma': 2.0,
    'use_amp': True,
    'seed': 42,
    'best_model_path': './best_relational_fusion_stanceosaurus.pt'
})

# An toàn: nếu lỡ bật use_evidence_injection=True mà không gắn dataset Excel, tự tắt lại thay vì lỗi.
if config.use_evidence_injection and not config.xlsx_dir:
    print('⚠️ use_evidence_injection=True nhưng không tìm thấy Excel gốc — tự động TẮT lại để tránh lỗi.')
    config.use_evidence_injection = False

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)
config.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'--> 🚀 Thiết bị: {config.device} | Max Epochs: {config.num_epochs} (early stop patience={config.early_stop_patience}) | Batch: {config.batch_size}x{config.accum_steps} | Evidence: {config.use_evidence_injection}')

In [ ]:
# ==============================================================================
# STEP 3: NẠP DỮ LIỆU (cùng input với 3 pipeline cũ) + GÁN NHÃN QUAN HỆ DISCOURSE (RULE-BASED)
# + EVIDENCE INJECTION KHI BẬT (mặc định TẮT, xem Cell 2)
# ==============================================================================
EXCEL_COLUMNS = ['number', 'date', 'status_id', 'in_reply_to_id', 'username', 'meta_info',
                  'claim', 'text', 'context', 'title', 'final_annotation', 'final_leaning', 'note_3']

def build_title_lookup(xlsx_filename):
    df = pd.read_excel(os.path.join(config.xlsx_dir, xlsx_filename))
    df.columns = EXCEL_COLUMNS
    lookup = {}
    for _, row in df.iterrows():
        title = str(row['title']).strip()
        if title and title.lower() != 'nan':
            lookup[row['status_id']] = title
    return lookup

# --- Bộ phân loại quan hệ discourse rule-based (thay thế LLM trong KAM của LLM-CRAN gốc) ---
CONTRASTIVE_MARKERS = {'but', 'however', 'although', 'though', 'yet', 'nevertheless', 'on the other hand', 'despite'}
CAUSAL_MARKERS = {'because', 'since', 'as a result', 'due to', 'therefore', 'so that', 'thus'}
SUCCESSION_MARKERS = {'then', 'next', 'after that', 'first', 'second', 'finally', 'meanwhile', 'also', 'and'}
RELATION_NAMES = ['contrastive', 'causal', 'succession', 'summary']  # fallback = 'summary'
RELATION_TO_ID = {name: i for i, name in enumerate(RELATION_NAMES)}

def classify_discourse_relation(prev_text, curr_text):
    """Heuristic rẻ tiền dựa discourse marker ở đầu câu hiện tại. Đây là surrogate OFFLINE của bước
    KAM trong LLM-CRAN gốc (vốn dùng GPT-4 để gán 'contrastive/causal/succession/summary').
    Thay bằng gọi LLM thật (GPT-4/Llama) nếu có API, sẽ chính xác hơn — ghi rõ trong báo cáo."""
    t = curr_text.strip().lower()
    first_words = ' '.join(t.split()[:4])
    for m in CONTRASTIVE_MARKERS:
        if t.startswith(m) or f' {m} ' in first_words:
            return RELATION_TO_ID['contrastive']
    for m in CAUSAL_MARKERS:
        if t.startswith(m) or f' {m} ' in first_words:
            return RELATION_TO_ID['causal']
    for m in SUCCESSION_MARKERS:
        if t.startswith(m):
            return RELATION_TO_ID['succession']
    return RELATION_TO_ID['summary']

def load_dialogue_json(json_filename, title_lookup):
    path = os.path.join(config.json_dir, json_filename)
    with open(path, 'r', encoding='utf-8') as f:
        samples = json.load(f)

    dialogues = []
    for s in samples:
        sentences = list(s['sentences'])
        speakers = list(s['speakers'])
        sid = s.get('status_id')
        title = title_lookup.get(sid) if config.use_evidence_injection else None
        if title:
            sentences = [title] + sentences
            speakers = [-1] + speakers

        n = len(sentences)
        # Đồ thị reply-chain: mỗi lượt i nối với lượt i-1 (quan hệ reply tuyến tính, đúng với cấu trúc
        # thread Twitter thực tế đã được dựng lại — không có branching info trong dữ liệu reconstructed).
        edges = [(i - 1, i) for i in range(1, n)]
        relations = [classify_discourse_relation(sentences[i-1], sentences[i]) for i in range(1, n)]

        dialogues.append({
            'target': str(s['target']).strip(),
            'sentences': sentences,
            'speakers': speakers,
            'edges': edges,
            'relations': relations,
            'label': int(s['label'])
        })
    return pd.DataFrame(dialogues)

if config.use_evidence_injection:
    print('[*] use_evidence_injection=True -> build title lookup (thử nghiệm RIÊNG, không dùng để so sánh công bằng với 3 pipeline cũ)...')
    train_titles = build_title_lookup('Train_final.xlsx')
    dev_titles = build_title_lookup('Dev_final.xlsx')
    test_titles = build_title_lookup('Test_final.xlsx')
else:
    print('[*] use_evidence_injection=False -> chỉ dùng train_conversations.json, giống hệt input của 3 pipeline cũ.')
    train_titles, dev_titles, test_titles = {}, {}, {}

print('[*] Nạp dialogue + xây đồ thị reply-chain + gán quan hệ discourse...')
train_df = load_dialogue_json('train_conversations.json', train_titles)
dev_df = load_dialogue_json('dev_conversations.json', dev_titles)
test_df = load_dialogue_json('test_conversations.json', test_titles)

if config.debug_mode:
    k = config.debug_samples_per_class
    train_df = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), k), random_state=config.seed)).reset_index(drop=True)
    dev_df = dev_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    test_df = test_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)

label_names = ['Supporting', 'Refuting', 'Other']
print(f'--> Train={len(train_df)} | Dev={len(dev_df)} | Test={len(test_df)}')
display(pd.DataFrame({'Train': train_df['label'].value_counts().sort_index()}, index=range(3)).set_axis(label_names))

In [ ]:
# ==============================================================================
# STEP 4: TOKENIZATION (mỗi lượt có target nối vào cuối, lấy biểu diễn bằng mean-pooling theo câu)
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

class RelationalStanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.data = df.to_dict('records')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        sentences, target = item['sentences'], item['target']
        encs = []
        for sen in sentences:
            text = f'{sen} . Stance target: {target}'
            enc = self.tokenizer(text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
            encs.append({'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0)})
        return {
            'encs': encs,
            'edges': item['edges'],
            'relations': item['relations'],
            'label': torch.tensor(item['label'], dtype=torch.long)
        }

def relational_collate_fn(batch):
    all_input_ids, all_att_masks, dia_idx = [], [], []
    edges_list, relations_list, labels_list = [], [], []
    curr = 0
    for sample in batch:
        n = len(sample['encs'])
        dia_idx.append((curr, curr + n))
        curr += n
        for e in sample['encs']:
            all_input_ids.append(e['input_ids'])
            all_att_masks.append(e['attention_mask'])
        edges_list.append(sample['edges'])
        relations_list.append(sample['relations'])
        labels_list.append(sample['label'])
    return {
        'input_ids': torch.stack(all_input_ids),
        'attention_mask': torch.stack(all_att_masks),
        'dia_idx': dia_idx,
        'edges': edges_list,
        'relations': relations_list,
        'labels': torch.stack(labels_list)
    }

train_dataset = RelationalStanceDataset(train_df, tokenizer, config.max_seq_len)
dev_dataset = RelationalStanceDataset(dev_df, tokenizer, config.max_seq_len)
test_dataset = RelationalStanceDataset(test_df, tokenizer, config.max_seq_len)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=relational_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=relational_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=relational_collate_fn)
print('--> ✅ DataLoader sẵn sàng!')

In [ ]:
# ==============================================================================
# STEP 5: KIẾN TRÚC — LKL (CNN) + CRL (GCN reply-chain) + LRL (RGCN discourse) + MULTI-HOP ATTENTION
# ==============================================================================
class LocalKnowledgeLayer(nn.Module):
    """CNN 1D cửa sổ trượt quanh lượt đang xét — nắm ngữ cảnh cục bộ gần nhất (LKL trong LLM-CRAN)."""
    def __init__(self, dim, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=pad)
        self.conv2 = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=pad)
        self.act = nn.GELU()

    def forward(self, H):          # H: (N, D)
        x = H.T.unsqueeze(0)       # (1, D, N)
        x = self.act(self.conv1(x))
        x = self.conv2(x)
        return x.squeeze(0).T      # (N, D)

class GCNLayer(nn.Module):
    """GCN 2 lớp trên đồ thị reply-chain — thay thế GRU tuần tự (CRL trong LLM-CRAN)."""
    def __init__(self, dim):
        super().__init__()
        self.w0 = nn.Linear(dim, dim)
        self.w1 = nn.Linear(dim, dim)
        self.act = nn.GELU()

    def forward(self, H, adj_norm):           # H: (N, D), adj_norm: (N, N) đã chuẩn hoá symmetric
        h = self.act(adj_norm @ self.w0(H))
        h = adj_norm @ self.w1(h)
        return h

class RGCNLayer(nn.Module):
    """RGCN 2 lớp trên đồ thị quan hệ discourse có kiểu cạnh (LRL trong LLM-CRAN, rút gọn 2 lớp)."""
    def __init__(self, dim, num_relations):
        super().__init__()
        self.num_relations = num_relations
        self.w_rel = nn.ModuleList([nn.Linear(dim, dim) for _ in range(num_relations)])
        self.w_self = nn.Linear(dim, dim)
        self.act = nn.GELU()

    def forward(self, H, rel_adj_list):   # rel_adj_list: list các (N,N) adjacency theo từng loại quan hệ (đã chuẩn hoá theo bậc)
        out = self.w_self(H)
        for r, adj_r in enumerate(rel_adj_list):
            out = out + self.w_rel[r](adj_r @ H)
        return self.act(out)

def build_adjacency(n, edges, device):
    """Đồ thị vô hướng (undirected) có self-loop, chuẩn hoá kiểu GCN: D^-1/2 (A+I) D^-1/2."""
    A = torch.eye(n, device=device)
    for i, j in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0
    deg = A.sum(dim=1)
    deg_inv_sqrt = torch.pow(deg.clamp(min=1e-6), -0.5)
    D_inv_sqrt = torch.diag(deg_inv_sqrt)
    return D_inv_sqrt @ A @ D_inv_sqrt

def build_relational_adjacency(n, edges, relations, num_relations, device):
    """Mỗi loại quan hệ discourse -> 1 ma trận kề riêng (có self-loop chia sẻ ở RGCNLayer.w_self)."""
    mats = [torch.zeros(n, n, device=device) for _ in range(num_relations)]
    for (i, j), r in zip(edges, relations):
        mats[r][i, j] = 1.0
        mats[r][j, i] = 1.0
    normed = []
    for A in mats:
        deg = A.sum(dim=1)
        deg_inv = torch.where(deg > 0, 1.0 / deg, torch.zeros_like(deg))
        normed.append(torch.diag(deg_inv) @ A)
    return normed

class MultiHopAttentionFusion(nn.Module):
    """Hợp nhất 3 luồng tri thức (Hl, Hs, Hr) có điều kiện theo vector lượt cuối (hxn), lặp p vòng
    (Multi-hop Attention trong LLM-CRAN, rút gọn còn 3 nhánh thay vì 4 vì không có Conversation-Act riêng)."""
    def __init__(self, dim, n_iters=2, lam=0.2):
        super().__init__()
        self.n_iters = n_iters
        self.lam = lam
        self.norm = nn.LayerNorm(dim)
        self.proj = nn.ModuleList([nn.Linear(dim, 1) for _ in range(3)])

    def forward(self, Hl, Hs, Hr, anchor_idx):
        streams = [Hl, Hs, Hr]
        for _ in range(self.n_iters):
            new_streams = []
            for k, Hk in enumerate(streams):
                anchor = Hk[anchor_idx].unsqueeze(0)                  # (1, D)
                scores = self.proj[k](Hk * anchor).squeeze(-1)        # (N,) — tương tác nhân với anchor
                c = torch.sigmoid(scores).unsqueeze(-1)               # (N, 1)
                update = self.norm(c * Hk)
                new_streams.append(Hk + self.lam * update)
            streams = new_streams
        return torch.cat([s[anchor_idx] for s in streams], dim=-1)    # (3*D,)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weights=None):
        super().__init__()
        self.gamma, self.weights = gamma, weights

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        pt = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()

counts = [sum(train_df['label'] == c) for c in range(config.num_classes)]
class_weights = torch.tensor([1.0 / (max(c, 1) ** 0.5) for c in counts], dtype=torch.float).to(config.device)
class_weights = class_weights / class_weights.sum()

class RelationalFusionModel(nn.Module):
    def __init__(self, config, class_weights):
        super().__init__()
        self.config = config
        self.backbone = AutoModel.from_pretrained(config.model_name)
        dim = self.backbone.config.hidden_size
        self.lkl = LocalKnowledgeLayer(dim, kernel_size=config.cnn_kernel)
        self.crl = GCNLayer(dim)
        self.lrl = RGCNLayer(dim, num_relations=config.num_relations_logical)
        self.fusion = MultiHopAttentionFusion(dim, n_iters=config.multihop_iters, lam=config.multihop_lambda)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(dim * 3, config.num_classes))
        self.focal_criterion = FocalLoss(gamma=config.focal_gamma, weights=class_weights)

    def encode_sentence_vectors(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # (M, L, D)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)   # mean-pooling theo câu
        return pooled   # (M, D)

    def forward(self, input_ids, attention_mask, dia_idx, edges, relations, labels=None):
        all_H = self.encode_sentence_vectors(input_ids, attention_mask)  # (sum_n, D)
        device = all_H.device

        logits_list = []
        for d_id, (st, ed) in enumerate(dia_idx):
            H = all_H[st:ed]            # (n, D)
            n = H.size(0)
            e, r = edges[d_id], relations[d_id]

            Hl = self.lkl(H)
            if n > 1:
                adj = build_adjacency(n, e, device)
                Hs = self.crl(H, adj)
                rel_adjs = build_relational_adjacency(n, e, r, self.config.num_relations_logical, device)
                Hr = self.lrl(H, rel_adjs)
            else:
                Hs, Hr = H, H

            fused = self.fusion(Hl, Hs, Hr, anchor_idx=n - 1)   # (3*D,) — n-1 = lượt cuối (đang xét)
            logits_list.append(fused)

        stance_features = torch.stack(logits_list)
        logits = self.classifier(stance_features)

        loss = None
        if labels is not None:
            loss = self.focal_criterion(logits, labels)
        return loss, logits

model = RelationalFusionModel(config, class_weights).to(config.device)
print(f'--> ✅ RelationalFusionModel khởi tạo xong! Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ==============================================================================
# STEP 6: OPTIMIZER (lr riêng cho backbone vs GCN/RGCN/fusion) & SCHEDULER
# ==============================================================================
backbone_params = list(model.backbone.parameters())
other_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
optimizer = AdamW([
    {'params': backbone_params, 'lr': config.learning_rate},
    {'params': other_params, 'lr': config.gnn_lr}
], weight_decay=config.weight_decay)

total_steps = (len(train_loader) // config.accum_steps) * config.num_epochs
warmup_steps = int(total_steps * config.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max(total_steps, 1))
scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp)
print(f'--> ✅ Tổng bước huấn luyện: {total_steps}')

In [ ]:
# ==============================================================================
# STEP 7: TRAINING LOOP
# ==============================================================================
def evaluate_model(model, loader, target_names):
    model.eval()
    all_preds, all_labels = [], []
    cls_list = list(range(len(target_names)))
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            labels = batch['labels'].to(config.device)
            with torch.cuda.amp.autocast(enabled=config.use_amp):
                _, logits = model(input_ids=input_ids, attention_mask=attention_mask, dia_idx=batch['dia_idx'],
                                   edges=batch['edges'], relations=batch['relations'], labels=None)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_preds, labels=cls_list, average='macro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    per_class_f1 = f1_score(all_labels, all_preds, labels=cls_list, average=None, zero_division=0)
    report = classification_report(all_labels, all_preds, labels=cls_list, target_names=target_names, zero_division=0, output_dict=True)
    return macro_f1, acc, per_class_f1, report

best_dev_f1 = 0.0
patience_counter = 0
print('\n[*] BẮT ĐẦU TRAINING (BERTweet + LKL/CRL/LRL + Multi-hop Attention Fusion)...')
for epoch in range(1, config.num_epochs + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    loop = tqdm(train_loader, desc=f'Epoch {epoch}/{config.num_epochs}')
    for step, batch in enumerate(loop):
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        labels = batch['labels'].to(config.device)

        with torch.cuda.amp.autocast(enabled=config.use_amp):
            loss, _ = model(input_ids=input_ids, attention_mask=attention_mask, dia_idx=batch['dia_idx'],
                             edges=batch['edges'], relations=batch['relations'], labels=labels)
            loss = loss / config.accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * config.accum_steps

        if (step + 1) % config.accum_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
        loop.set_postfix({'loss': f"{total_loss/(step+1):.3f}"})

    avg_train_loss = total_loss / len(train_loader)
    dev_f1, dev_acc, _, _ = evaluate_model(model, dev_loader, label_names)
    print(f'--> [Epoch {epoch}] Train Loss: {avg_train_loss:.4f} | Dev Macro-F1: {dev_f1*100:.2f}% | Dev Acc: {dev_acc*100:.2f}%')

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        patience_counter = 0
        torch.save(model.state_dict(), config.best_model_path)
        print(f'    🏆 [Checkpoint tốt nhất] Dev F1: {best_dev_f1*100:.2f}%')
    else:
        patience_counter += 1
        print(f'    ⏳ [Không cải thiện] {patience_counter}/{config.early_stop_patience} epoch liên tiếp (Best: {best_dev_f1*100:.2f}%)')
        if patience_counter >= config.early_stop_patience:
            print(f'    🛑 [EARLY STOPPING] Dừng tại epoch {epoch} vì Dev Macro-F1 không cải thiện sau {config.early_stop_patience} epoch liên tiếp.')
            break

In [ ]:
# ==============================================================================
# STEP 8: ĐÁNH GIÁ TEST + ĐỐI SÁNH
# ==============================================================================
if os.path.exists(config.best_model_path):
    model.load_state_dict(torch.load(config.best_model_path, map_location=config.device))
    print(f'--> [✓] Đã nạp checkpoint tốt nhất.')

test_f1, test_acc, test_per_f1, test_report_dict = evaluate_model(model, test_loader, label_names)

model_name = 'Relational Knowledge Fusion'
if config.use_evidence_injection:
    model_name += ' + Evidence [KHÔNG cùng input với 3 pp cũ!]'
model_name += ' (P2, Ours)'

res_dict = {'Model': model_name, 'Mode': 'DEBUG' if config.debug_mode else 'FULL',
            'Same_input_as_3_pipeline_cu': not config.use_evidence_injection,
            'Accuracy': round(test_acc * 100, 2), 'Macro_F1': round(test_f1 * 100, 2)}
for idx, name in enumerate(label_names):
    if idx < len(test_per_f1):
        res_dict[f'F1_{name}'] = round(test_per_f1[idx] * 100, 2)

results_df = pd.DataFrame([res_dict])
results_df.to_csv('stanceosaurus_p2_relational_fusion_results.csv', index=False)
print('\n--- 📊 KẾT QUẢ P2 ---')
display(results_df)

if config.use_evidence_injection:
    print('\n⚠️ LƯU Ý: use_evidence_injection=True -> kết quả này KHÔNG dùng input giống 3 pipeline cũ.')
    print('   Không nên so sánh thẳng dòng này với bảng tham chiếu. Hãy so với kết quả chạy')
    print('   use_evidence_injection=False (cùng notebook) để biết riêng title đóng góp bao nhiêu.')
else:
    reference_rows = [
        {'Model': 'BERTweet baseline (Focal, EMNLP2022)', 'Macro_F1': 57.25, 'F1_Supporting': 56.05, 'F1_Refuting': 47.97, 'F1_Other': 67.71},
        {'Model': 'SITPCL (ACL2025 Findings)', 'Macro_F1': 43.14, 'F1_Supporting': 34.79, 'F1_Refuting': 34.49, 'F1_Other': 60.13},
        {'Model': 'SITPCL-FoT + Sparsemax (cứng)', 'Macro_F1': 36.98, 'F1_Supporting': 37.59, 'F1_Refuting': 32.91, 'F1_Other': 40.45},
    ]
    compare_df = pd.concat([pd.DataFrame(reference_rows), results_df], ignore_index=True)
    print('\n🔥 BẢNG ĐỐI SÁNH CÔNG BẰNG (CÙNG INPUT train_conversations.json):')
    display(compare_df)
    compare_df.to_csv('stanceosaurus_p2_final_comparison.csv', index=False)

report_df = pd.DataFrame(test_report_dict).T
display(report_df.round(3))

if config.debug_mode:
    print('\n🎉 DEBUG MODE chạy OK! Quay lại Cell 2 đổi DEBUG_MODE=False rồi Save & Run All.')
else:
    print('\n✅ ĐÃ CHẠY XONG FULL DỮ LIỆU P2!')